In [ ]:
# Figure 5 - Global distribution of serotype 2 genomes by sequence type
#
# This figure is built from Supplementary Data 1, which is included in this
# repository; the same counts are also in the "Figure 5" sheet of Source Data.
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np
from matplotlib.patches import Wedge, Patch

GENOMES = "data/Supplementary_Data_1.xlsx"
BASEMAP = "data/naturalearth_lowres/ne_110m_admin_0_countries.shp"
OUT     = "results/figures/Figure_05.pdf"

# GeoPandas >= 1.0 removed gpd.datasets. Download the Natural Earth 1:110m
# "Admin 0 - Countries" shapefile (public domain, naturalearthdata.com)
# into data/naturalearth_lowres/ before running this notebook.


In [ ]:
IMPORTANT_STS = ["74", "1504", "3744", "5083", "5199", "9476"]

g = pd.read_excel(GENOMES, sheet_name="Sheet1")
g["ST"] = g["ST"].astype(str)
g["ST"] = g["ST"].apply(lambda x: x if x in IMPORTANT_STS else "Others")

cols  = IMPORTANT_STS + ["Others"]
pivot = (g.groupby(["Country", "ST"]).size().reset_index(name="count")
           .pivot(index="Country", columns="ST", values="count")
           .fillna(0).astype(int))
for c in cols:
    if c not in pivot.columns:
        pivot[c] = 0
pivot = pivot[cols]
pivot["Total"] = pivot.sum(axis=1)
df = pivot.reset_index()
print(f"{int(df['Total'].sum())} genomes across {len(df)} countries")


In [ ]:
worldmap = gpd.read_file(BASEMAP)
name_col = "NAME" if "NAME" in worldmap.columns else "name"
worldmap["Longitude"] = worldmap.geometry.representative_point().x
worldmap["Latitude"]  = worldmap.geometry.representative_point().y

df = df.merge(worldmap[[name_col, "Longitude", "Latitude"]],
              how="left", left_on="Country", right_on=name_col).drop(columns=[name_col])
df = df[~df["Country"].isin(["Others", "Unknown"])]

missing = df[df["Longitude"].isna()]["Country"].tolist()
if missing:
    print("Not matched to the basemap, check spelling:", missing)


In [ ]:
def draw_pie(ax, ratios, x, y, size, colors):
    start = 0
    for ratio, colour in zip(ratios, colors):
        end = start + ratio * 360
        ax.add_patch(Wedge((x, y), size, start, end, color=colour, alpha=0.8))
        start = end

COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
          "#9467bd", "#8c564b", "#7f7f7f"]

fig, ax = plt.subplots(figsize=(32, 20))
worldmap.plot(ax=ax, edgecolor="#91948a", facecolor="white")

for _, row in df.dropna(subset=["Longitude"]).iterrows():
    total = row["Total"]
    if total > 0:
        ratios = [row[st] / total for st in cols]
        draw_pie(ax, ratios, row["Longitude"], row["Latitude"], np.sqrt(total), COLORS)
        ax.text(row["Longitude"] + 3, row["Latitude"] + 3,
                row["Country"], fontsize=20, ha="center")

handles = [Patch(color=COLORS[i], label=cols[i]) for i in range(len(cols))]
plt.xlim([-180, 180]); plt.ylim([-60, 90])
plt.legend(handles=handles, title="STs", loc="lower left")
plt.xlabel("Longitude", fontsize=20); plt.ylabel("Latitude", fontsize=20)
plt.savefig(OUT, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
